# 1. Формирование FM/таргетов



In [ ]:
# ===== НАСТРОЙКИ ЗАПУСКА =====
RUN_MODE = 'monthly_oct_valid_dec_inference'
TARGET_MONTH = '2026-10-01'
MIN_PERIOD_CFG = 202505
MAX_PERIOD_CFG = 202610
DATASET_WITH_FEATURES_CFG = 'sandbox_crft.mae_nbc_dataset_with_features_20250901_20261101'
DATASET_WITH_FEATURES_DYNAMIC_CFG = 'sandbox_crft.mae_nbc_dataset_with_features_fs_nbc_monthly_202610'
DATASET_UPLIFT_BINARY_CFG = 'sandbox_crft.mae_nbc_dataset_with_features_fm_delta_new_scenarios'
DATASET_UPLIFT_BINARY_DYNAMIC_CFG = 'sandbox_crft.mae_nbc_dataset_with_features_fm_delta_modular_full'
if RUN_MODE != 'monthly_oct_valid_dec_inference' or TARGET_MONTH != '2026-10-01':
    raise ValueError('Для нового месяца надо пересчитать и проверить источники/периоды; простая смена TARGET_MONTH недостаточна.')

VALID_MONTH_CFG = "2026-10-01"
TEST_MONTH_CFG = None
INFERENCE_MONTH_CFG = "2026-12-01"


# Import and connect

In [ ]:
# Единая папка common для файлов/конфигов, экспортируемых к обучению.
from pathlib import Path
import os, sys
_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd / 'common', _cwd.parent / 'common']
PROJECT_ROOT = next((x for x in _candidates if (x / 'srcc' / 'nbc_project').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Нет common/srcc/nbc_project рядом с ноутбуком')
os.chdir(PROJECT_ROOT)
SRC_PATH = PROJECT_ROOT / 'srcc'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print('COMMON_PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
from rail_connectors.connection import connect
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd
pd.set_option('display.max_columns', None)

import gc

conn_impala = connect(kerberos = {'use_credentials': True}, to='IMPALA')
conn_hive = connect(kerberos = {'use_credentials': True}, to='DATALAKE')
conn = conn_impala


# Импорт фичей 
from nbc_project.feature_lists import load_feature_lists, get_feature_list, select_existing_columns
from nbc_project.data_fm_helpers import (
    pyarrow_to_pandas,
    pyarrow_random_sample_to_pandas,
    save_optimized_csv,
    load_optimize_save_dataset,
    unique_features,
    create_selected_features_table,
)

FEATURE_LISTS = load_feature_lists()
DROP_COLS = get_feature_list('data_fm_drop_columns')
MUST_HAVE = get_feature_list('data_fm_must_have_features')

In [ ]:
%%time

if conn == conn_impala:
    print("Connect Impala")
    source_tables = [
        'sandbox_crft.pos_trans_1_pl_mae' 
    ]
    
    with conn:
        for table in source_tables:
            conn.execute(f"invalidate metadata {table}")

else:
    print("Connect Hive")

# Const

In [ ]:
POS_SEND_M = "sandbox_crft.mae_pos_send_m_delta" 
M_TARGET_BASE = "sandbox_crft.mae_pos_m_delta_target_base"
M_TARGET_BINS = "sandbox_crft.mae_pos_m_delta_target_bins"
M_TARGET_FLAG = "sandbox_crft.mae_pos_m_delta_target_flag"
M_TARGET_FLAG_CDI = "sandbox_crft.mae_pos_m_delta_target_flag_cdi"

hist_m = 1 # количество месяцев, на которых считаем средний исторический POS
MIN_PERIOD = MIN_PERIOD_CFG #минимальный расчетный период
MAX_PERIOD = MAX_PERIOD_CFG #максимальный расчетный период
N_BINS = 500

# Source

In [ ]:
%%time

query = f"""
    select *
    from sandbox_crft.pos_trans_1_pl_mae 
    limit 1
"""

with conn:
    temp = conn.fetch(query)

temp	

# Main

## POS_SEND_M

### Create

In [ ]:
def yyyymm_to_date(period: int) -> datetime:
    return datetime.strptime(str(period), "%Y%m")

def date_to_yyyymm(dt: datetime) -> int:
    return int(dt.strftime("%Y%m"))

min_period_dt = yyyymm_to_date(MIN_PERIOD)
max_period_dt = yyyymm_to_date(MAX_PERIOD)

min_trans_dt = min_period_dt - relativedelta(months=hist_m + 1)
max_trans_dt = max_period_dt

MIN_TRANS_PERIOD = date_to_yyyymm(min_trans_dt)
MAX_TRANS_PERIOD = date_to_yyyymm(max_trans_dt)

print(MIN_TRANS_PERIOD, MAX_TRANS_PERIOD)

In [ ]:
%%time

query = f"""
    CREATE TABLE {POS_SEND_M}
    STORED AS PARQUET  
    TBLPROPERTIES('transactional' = 'false')
    AS

    select 
        TRUNC(CAST(cast(trans_mm as date) AS DATE), 'MM')  as period_dt
        , customer_id
        , sum(abs(charge_rub_amt)) as sum_charge_rub_amt
    from sandbox_crft.pos_trans_1_pl_mae
    where 1=1
        and year(cast(trans_mm as date)) * 100 + month(cast(trans_mm as date)) between {MIN_TRANS_PERIOD} and {MAX_TRANS_PERIOD}
        and customer_id is not null
    group by 
        TRUNC(CAST(cast(trans_mm as date) AS DATE), 'MM')
        , customer_id
"""

with conn:
    conn.execute(f"drop table if exists {POS_SEND_M}")
    conn.execute(query)	

### Check

In [ ]:
%%time

query = f"""
    select *
    from {POS_SEND_M}
    limit 3
"""

with conn:
    temp = conn.fetch(query)

temp	

In [ ]:
%%time

query = f"""
    select min(period_dt), max(period_dt)
    from {POS_SEND_M}
    limit 1
"""

with conn:
    temp = conn.fetch(query)

temp	

## M_TARGET_BASE

### Create

In [ ]:
%%time

query = f"""
    CREATE TABLE {M_TARGET_BASE}
    STORED AS PARQUET  
    TBLPROPERTIES('transactional' = 'false')
    AS

    WITH calc_periods AS (
        SELECT DISTINCT
            period_dt
        FROM {POS_SEND_M}
        WHERE year(period_dt) * 100 + month(period_dt)
              BETWEEN {MIN_PERIOD} AND {MAX_PERIOD}
    ),
    
    cur AS (
        SELECT
            period_dt,
            customer_id,
            sum_charge_rub_amt AS m_cur
        FROM {POS_SEND_M}
        WHERE year(period_dt) * 100 + month(period_dt)
              BETWEEN {MIN_PERIOD} AND {MAX_PERIOD}
    ),
    
    hist AS (
        SELECT
            p.period_dt,
            t.customer_id,
            SUM(t.sum_charge_rub_amt) AS m_hist_sum
        FROM calc_periods p
        JOIN {POS_SEND_M} t
          ON t.period_dt >= ADD_MONTHS(p.period_dt, -({hist_m} + 1))
         AND t.period_dt <= ADD_MONTHS(p.period_dt, -2)
        GROUP BY
            p.period_dt,
            t.customer_id
    ),
    
    keys AS (
        SELECT period_dt, customer_id FROM cur
        UNION
        SELECT period_dt, customer_id FROM hist
    )
    
    SELECT
        k.period_dt,
        k.customer_id,
        COALESCE(c.m_cur, 0) AS m_cur,
        COALESCE(h.m_hist_sum, 0) AS m_hist_sum,
        COALESCE(h.m_hist_sum, 0) / {hist_m}.0 AS m_hist_avg
    FROM keys k
    LEFT JOIN cur c
        ON k.period_dt = c.period_dt
       AND k.customer_id = c.customer_id
    LEFT JOIN hist h
        ON k.period_dt = h.period_dt
       AND k.customer_id = h.customer_id  
"""

with conn:
    conn.execute(f"drop table if exists {M_TARGET_BASE}")
    conn.execute(query)	

### Check

In [ ]:
%%time

query = f"""
    select *
    from {M_TARGET_BASE}
    limit 1
"""

with conn:
    temp = conn.fetch(query)

temp	

In [ ]:
%%time

query = f"""
    select *
    from {M_TARGET_BASE}
    where m_hist_sum = 0
    limit 1
"""

with conn:
    temp = conn.fetch(query)

temp	

## M_TARGET_BINS

### Create

In [ ]:
%%time

query = f"""
    CREATE TABLE {M_TARGET_BINS}
    STORED AS PARQUET  
    TBLPROPERTIES('transactional' = 'false')
    AS

    WITH base AS (
        SELECT
            period_dt,
            customer_id,
            m_cur,
            m_hist_sum,
            m_hist_avg
        FROM {M_TARGET_BASE}
    ),
    
    cur_pos AS (
        SELECT
            period_dt,
            customer_id,
            NTILE({N_BINS}) OVER (
                PARTITION BY period_dt
                ORDER BY m_cur
            ) AS cur_bin_pos
        FROM base
        WHERE m_cur > 0
    ),
    hist_pos AS (
        SELECT
            period_dt,
            customer_id,
            NTILE({N_BINS}) OVER (
                PARTITION BY period_dt
                ORDER BY m_hist_avg
            ) AS hist_bin_pos
        FROM base
        WHERE m_hist_avg > 0
    )
    SELECT
        b.period_dt,
        b.customer_id,
        b.m_cur,
        b.m_hist_sum,
        b.m_hist_avg,
        CASE
            WHEN b.m_cur = 0 THEN 0
            ELSE cp.cur_bin_pos
        END AS cur_bin,
        CASE
            WHEN b.m_hist_avg = 0 THEN 0
            ELSE hp.hist_bin_pos
        END AS hist_bin
    FROM base b
    LEFT JOIN cur_pos cp
        ON b.period_dt = cp.period_dt
       AND b.customer_id = cp.customer_id
    LEFT JOIN hist_pos hp
        ON b.period_dt = hp.period_dt
       AND b.customer_id = hp.customer_id
"""

with conn:
    conn.execute(f"drop table if exists {M_TARGET_BINS}")
    conn.execute(query)	

### Check

In [ ]:
%%time

query = f"""
    select *
    from {M_TARGET_BINS}
    limit 1
"""

with conn:
    temp = conn.fetch(query)

temp	

In [ ]:
%%time

query = f"""
    select period_dt, count(*)
    from {M_TARGET_BINS}
    group by period_dt
    order by period_dt
    limit 10
"""

with conn:
    temp = conn.fetch(query)

temp	

In [ ]:
%%time

query = f"""
    select period_dt, count(*)
    from {M_TARGET_BINS}
    group by period_dt, hist_bin
    order by period_dt, hist_bin
    limit 10
"""

with conn:
    temp = conn.fetch(query)

temp	

In [ ]:
%%time

query = f"""
    select period_dt, hist_bin, count(*)
    from {M_TARGET_BINS}
    group by period_dt, hist_bin
    order by period_dt, hist_bin
    limit 10
"""

with conn:
    temp = conn.fetch(query)

temp	

## M_TARGET_FLAG

### Create

In [ ]:
%%time

query = f"""
    CREATE TABLE {M_TARGET_FLAG}
    STORED AS PARQUET  
    TBLPROPERTIES('transactional' = 'false')
    AS
    SELECT
        period_dt,
        customer_id,
        m_cur,
        m_hist_sum,
        m_hist_avg,
        cur_bin,
        hist_bin,
        cur_bin - hist_bin AS delta_bin,
        CASE
            WHEN cur_bin > hist_bin THEN 1
            ELSE 0
        END AS target_flag
    FROM {M_TARGET_BINS}
"""

with conn:
    conn.execute(f"drop table if exists {M_TARGET_FLAG}")
    conn.execute(query)	

### Check

In [ ]:
%%time

query = f"""
    select *
    from {M_TARGET_FLAG}
    limit 3
"""

with conn:
    temp = conn.fetch(query)

temp	

In [ ]:
%%time

query = f"""
    SELECT
        period_dt,
        target_flag,
        COUNT(*) AS cnt
    FROM {M_TARGET_FLAG}
    GROUP BY
        period_dt,
        target_flag
    ORDER BY
        period_dt,
        target_flag
"""

with conn:
    temp = conn.fetch(query)

temp	

## M_TARGET_FLAG_CDI

### Create

In [ ]:
%%time

query = f"""
    CREATE TABLE {M_TARGET_FLAG_CDI}
    STORED AS PARQUET  
    TBLPROPERTIES('transactional' = 'false')
    AS

    select 
        np.period_dt
        , np.customer_id
        , np.target_flag
        , delta_bin
        , hist_bin
        , cast(rc.customer_src_cd as bigint) as cdi_id
    from {M_TARGET_FLAG} np
    join dm_dcp_fl.ref_customer rc
        on rc.customer_id = np.customer_id 
        and rc.exp_dt > now()
        and rc.source_id = 3
        and cust_birth_dt is not null
        and cust_doc_issue_dt is not null
    -- where np.target_flag = 1
"""

with conn:
    conn.execute(f"drop table if exists {M_TARGET_FLAG_CDI}")
    conn.execute(query)	

### Check

In [ ]:
%%time

query = f"""
    select *
    from {M_TARGET_FLAG_CDI}
    limit 1
"""

with conn:
    temp = conn.fetch(query)

temp	

In [ ]:
%%time

query = f"""
    select period_dt, count(*), count(distinct customer_id), count(distinct cdi_id)
    from {M_TARGET_FLAG_CDI}
    group by period_dt
    order by period_dt
    limit 10
"""

with conn:
    temp = conn.fetch(query)

temp	

In [ ]:
# CURRENT branch: старый feature store acrm.common_segmentation.
DATASET_WITH_FEATURES = DATASET_WITH_FEATURES_CFG

In [ ]:
%%time

query = f"""
    select forecast_month, count(*), count(distinct cdi_id)
    from {DATASET_WITH_FEATURES}
    group by forecast_month
    order by forecast_month
"""

with conn:
    temp = conn.fetch(query)

temp

# JOIN с фичами из data_collection

In [ ]:
DATASET_UPLIFT_BINARY = DATASET_UPLIFT_BINARY_CFG

In [ ]:
%%time

query = f"""
    CREATE TABLE {DATASET_UPLIFT_BINARY}
    STORED AS PARQUET  
    TBLPROPERTIES('transactional' = 'false')
    AS
    select period_dt, hist_bin, delta_bin, target_flag, f.*
    from {M_TARGET_FLAG_CDI} t
    join {DATASET_WITH_FEATURES} f
    on f.forecast_month = t.period_dt
    and f.cdi_id = cast(t.cdi_id as bigint)
    where forecast_month >= '2025-09-01'
"""

with conn:
    conn.execute(f"drop table if exists {DATASET_UPLIFT_BINARY}")
    conn.execute(query)

In [ ]:
%%time

query = f"""
    select 
        sum(case when target_loyalty_connected_flg = 1 then 1 else 0 end) as loyal_cnt, 
        sum(case when target_loyalty_connected_flg = 0 then 1 else 0 end) as noloyal_cnt
    from {DATASET_UPLIFT_BINARY}
"""

with conn:
    temp = conn.fetch(query)

temp
# Октябрь ОБЯЗАТЕЛЬНО должен присутствовать в current target-срезе.
with conn:
    _oct = conn.fetch(f"SELECT COUNT(*) AS n FROM {DATASET_UPLIFT_BINARY} WHERE cast(forecast_month AS DATE) = '{TARGET_MONTH}'")
assert int(_oct.iloc[0, 0]) > 0, 'Нет октября в current target dataset'
print('Current October target rows:', int(_oct.iloc[0, 0]))


In [ ]:
# Не храним учётные данные в ноутбуке; используем подключение через Kerberos.
# Если на сервисном пользователе нужны особые права для CTAS, запросите их штатным способом.
conn_hive = connect(kerberos={'use_credentials': True}, to='IMPALA')


In [ ]:
LOYALTY_NBC_DATASET = 'sandbox_crft.mae_nbc_fm_loyalty_dataset_new_scenarios'
NOLOYALTY_NBC_DATASET = 'sandbox_crft.mae_nbc_fm_noloyalty_dataset_new_scenarios'
NOLOYALTY_TEMP = 'sandbox_crft.mae_control_ranked_temp'

In [ ]:
with conn:
    conn.execute(f'''drop table if exists {LOYALTY_NBC_DATASET}''')
    conn.execute(f'''drop table if exists {NOLOYALTY_NBC_DATASET}''')
    conn.execute(f'''drop table if exists {NOLOYALTY_TEMP}''')

In [ ]:
%%time

with conn_hive:
    conn_hive.execute(f'''
    CREATE TABLE {LOYALTY_NBC_DATASET}
    -- PARTITIONED BY (title_name)
    STORED AS PARQUET
    TBLPROPERTIES('transactional' = 'false')
    AS (
        select *
        from {DATASET_UPLIFT_BINARY}
        where target_loyalty_connected_flg = 1
        )
    ''')

In [ ]:
%%time

with conn_hive:
    conn_hive.execute(f'''
        create table {NOLOYALTY_TEMP}
        stored as parquet
        tblproperties('transactional'='false')
        as
        with treat_counts as (
            select
                forecast_month,
                hist_bin,
                count(*) as n_treat
            from {DATASET_UPLIFT_BINARY}
            where target_loyalty_connected_flg = 1
            group by forecast_month, hist_bin
        ),
        not_loyalty as (
            select *
            from {DATASET_UPLIFT_BINARY}
            where target_loyalty_connected_flg = 0
        ),
        control_ranked as (
            select
                c.*,
                t.n_treat,
                row_number() over (
                    partition by c.forecast_month, c.hist_bin
                    order by rand()
                ) as rn
            from not_loyalty c
            join treat_counts t
                on c.forecast_month = t.forecast_month
               and c.hist_bin   = t.hist_bin
        )
        select *
        from control_ranked;
        ''')

In [ ]:
%%time

with conn_hive:
    conn_hive.execute(f'''
    CREATE TABLE {NOLOYALTY_NBC_DATASET}
    -- PARTITIONED BY (title_name)
    STORED AS PARQUET
    TBLPROPERTIES('transactional' = 'false')
    AS (
        select *
        from {NOLOYALTY_TEMP}
        where rn <= 2 * n_treat
    )
    ''')

## Dynamic FS source — modular feature store

`current` остаётся на старом источнике и старых статических feature sets. Здесь создаются отдельные full-feature T/C candidates из `feature_store_inc_modular`. Row keys берём из current T/C, поэтому в benchmark меняется feature store/feature selection, а не клиентская выборка.


In [ ]:
DATASET_WITH_FEATURES_DYNAMIC = DATASET_WITH_FEATURES_DYNAMIC_CFG
DATASET_UPLIFT_BINARY_DYNAMIC = DATASET_UPLIFT_BINARY_DYNAMIC_CFG

with conn:
    conn.execute(f"drop table if exists {DATASET_UPLIFT_BINARY_DYNAMIC}")
    conn.execute(f"""
        CREATE TABLE {DATASET_UPLIFT_BINARY_DYNAMIC}
        STORED AS PARQUET
        TBLPROPERTIES('transactional' = 'false')
        AS
        select period_dt, hist_bin, delta_bin, target_flag, f.*
        from {M_TARGET_FLAG_CDI} t
        join {DATASET_WITH_FEATURES_DYNAMIC} f
          on f.forecast_month = t.period_dt
         and f.cdi_id = cast(t.cdi_id as bigint)
        where f.forecast_month >= '2025-09-01'
    """)

DATASET_UPLIFT_BINARY_DYNAMIC


In [ ]:
LOYALTY_NBC_DATASET_DYNAMIC = 'sandbox_crft.mae_nbc_fm_loyalty_dataset_modular_full'
NOLOYALTY_NBC_DATASET_DYNAMIC = 'sandbox_crft.mae_nbc_fm_noloyalty_dataset_modular_full'

with conn:
    conn.execute(f"drop table if exists {LOYALTY_NBC_DATASET_DYNAMIC}")
    conn.execute(f"drop table if exists {NOLOYALTY_NBC_DATASET_DYNAMIC}")

    conn.execute(f"""
        CREATE TABLE {LOYALTY_NBC_DATASET_DYNAMIC}
        STORED AS PARQUET AS
        WITH current_keys AS (
            SELECT DISTINCT cdi_id, forecast_month
            FROM {LOYALTY_NBC_DATASET}
        )
        SELECT d.*
        FROM {DATASET_UPLIFT_BINARY_DYNAMIC} d
        JOIN current_keys k
          ON d.cdi_id = k.cdi_id
         AND d.forecast_month = k.forecast_month
    """)

    conn.execute(f"""
        CREATE TABLE {NOLOYALTY_NBC_DATASET_DYNAMIC}
        STORED AS PARQUET AS
        WITH current_keys AS (
            SELECT DISTINCT cdi_id, forecast_month
            FROM {NOLOYALTY_NBC_DATASET}
        )
        SELECT d.*
        FROM {DATASET_UPLIFT_BINARY_DYNAMIC} d
        JOIN current_keys k
          ON d.cdi_id = k.cdi_id
         AND d.forecast_month = k.forecast_month
    """)

    dynamic_rows_check = conn.fetch(f"""
        SELECT 'current_T' source, COUNT(*) rows_cnt FROM {LOYALTY_NBC_DATASET}
        UNION ALL SELECT 'modular_T', COUNT(*) FROM {LOYALTY_NBC_DATASET_DYNAMIC}
        UNION ALL SELECT 'current_C', COUNT(*) FROM {NOLOYALTY_NBC_DATASET}
        UNION ALL SELECT 'modular_C', COUNT(*) FROM {NOLOYALTY_NBC_DATASET_DYNAMIC}
    """)

counts = dict(zip(dynamic_rows_check['source'], dynamic_rows_check['rows_cnt']))
assert counts['current_T'] == counts['modular_T'], f"T row mismatch: {counts}"
assert counts['current_C'] == counts['modular_C'], f"C row mismatch: {counts}"
dynamic_rows_check

with conn:
    _oct_d = conn.fetch(f"SELECT COUNT(*) AS n FROM {LOYALTY_NBC_DATASET_DYNAMIC} WHERE cast(forecast_month AS DATE) = '{TARGET_MONTH}'")
assert int(_oct_d.iloc[0, 0]) > 0, 'Нет октября в dynamic loyalty dataset'
print('Dynamic October target rows:', int(_oct_d.iloc[0, 0]))


In [ ]:
!rm -rf ./data/datasets/mae_nbc_fm_loyalty_dataset_modular_full
!rm -rf ./data/datasets/mae_nbc_fm_noloyalty_dataset_modular_full

!hdfs dfs -copyToLocal /warehouse/tablespace/managed/hive/sandbox_crft.db/mae_nbc_fm_loyalty_dataset_modular_full/ ./data/datasets
!hdfs dfs -copyToLocal /warehouse/tablespace/managed/hive/sandbox_crft.db/mae_nbc_fm_noloyalty_dataset_modular_full/ ./data/datasets


In [ ]:
print(LOYALTY_NBC_DATASET, NOLOYALTY_NBC_DATASET, sep='\n')

In [ ]:
!rm -rf ./data/datasets/mae_nbc_fm_loyalty_dataset_new_scenarios

In [ ]:
!rm -rf ./data/datasets/mae_nbc_fm_noloyalty_dataset_new_scenarios

In [ ]:
!hdfs dfs -copyToLocal /warehouse/tablespace/external/hive/sandbox_crft.db/mae_nbc_fm_loyalty_dataset_new_scenarios/ ./data/datasets

In [ ]:
!hdfs dfs -copyToLocal /warehouse/tablespace/external/hive/sandbox_crft.db/mae_nbc_fm_noloyalty_dataset_new_scenarios/ ./data/datasets

# Сохранение датасетов для feature selection

In [ ]:
# Технические колонки для удаления вынесены в configs/feature_lists.json
# DROP_COLS = get_feature_list('data_fm_drop_columns')

## Для лояльных

In [ ]:
# %%time

# df_loyalty = pyarrow_random_sample_to_pandas(
#     path=f"./data/datasets/mae_nbc_fm_loyalty_dataset_new_scenarios",
#     sample_frac=0.12,
#     max_rows=500_000,
#     batch_size=50_000
# )

In [ ]:
# df_loyalty

In [ ]:
# !mkdir ./data/files
# df_loyalty = save_optimized_csv(df_loyalty, './data/files/df_loyalty_for_featsel.csv', drop_columns=DROP_COLS, keep_columns=['n_treat', 'rn'])

In [ ]:
if "df_loyalty" in globals():
    del df_loyalty

## Для нелояльных

In [ ]:
# %%time

# df_noloyalty = pyarrow_random_sample_to_pandas(
#     path=f"./data/datasets/mae_nbc_fm_noloyalty_dataset_new_scenarios",
#     sample_frac=0.05,
#     max_rows=200_000,
#     batch_size=50_000
# )
# df_noloyalty

In [ ]:
# df_noloyalty = save_optimized_csv(df_noloyalty, './data/files/df_noloyalty_for_featsel.csv', drop_columns=DROP_COLS)

# Сохранение датасетов после feature selection для всех моделей uplift

In [ ]:
from pathlib import Path
import gc
import pyarrow.dataset as ds
import pandas as pd


def save_parquet_dataset_to_csv_in_batches(
    input_path: str,
    output_csv_path: str,
    columns: list[str] | None = None,
    batch_size: int = 200_000,
) -> None:
    input_path = Path(input_path)
    output_csv_path = Path(output_csv_path)

    output_csv_path.parent.mkdir(parents=True, exist_ok=True)

    if output_csv_path.exists():
        output_csv_path.unlink()

    dataset = ds.dataset(str(input_path), format="parquet")

    first_batch = True
    rows_written = 0

    scanner = dataset.scanner(
        columns=columns,
        batch_size=batch_size,
    )

    for batch_num, batch in enumerate(scanner.to_batches(), start=1):
        df_part = batch.to_pandas()

        # лёгкая оптимизация числовых колонок
        for col in df_part.select_dtypes(include=["int64"]).columns:
            df_part[col] = pd.to_numeric(df_part[col], downcast="integer")

        for col in df_part.select_dtypes(include=["float64"]).columns:
            df_part[col] = pd.to_numeric(df_part[col], downcast="float")

        df_part.to_csv(
            output_csv_path,
            mode="w" if first_batch else "a",
            header=first_batch,
            index=False,
        )

        rows_written += len(df_part)
        print(f"batch={batch_num}, rows_written={rows_written:,}")

        first_batch = False

        del df_part, batch
        gc.collect()

    print(f"Готово. CSV сохранён: {output_csv_path}")
    print(f"Всего строк: {rows_written:,}")

In [ ]:
# Must-have фичи вынесены в configs/feature_lists.json
MUST_HAVE = get_feature_list('data_fm_must_have_features')
MUST_HAVE[:5], len(MUST_HAVE)

## POS

In [ ]:
# Список выбранных фичей вынесен в configs/feature_lists.json
SELECT_POS_LOYAL_COLS = get_feature_list('data_fm_pos_loyalty_columns')
SELECT_POS_LOYAL_COLS[:5], len(SELECT_POS_LOYAL_COLS)

In [ ]:
POS_LOYAL_FEATURES = unique_features(MUST_HAVE, SELECT_POS_LOYAL_COLS)
len(POS_LOYAL_FEATURES), POS_LOYAL_FEATURES[:5]

In [ ]:
%%time

create_selected_features_table(
    conn=conn_hive,
    output_table='sandbox_crft.mae_nbc_pos_loyalty_dataset_select_feats_new_scenarios',
    source_table='sandbox_crft.mae_nbc_fm_loyalty_dataset_new_scenarios',
    features=POS_LOYAL_FEATURES,
)

In [ ]:
# Список выбранных фичей вынесен в configs/feature_lists.json
SELECT_POS_NOLOYAL_COLS = get_feature_list('data_fm_pos_noloyalty_columns')
SELECT_POS_NOLOYAL_COLS[:5], len(SELECT_POS_NOLOYAL_COLS)

In [ ]:
POS_NOLOYAL_FEATURES = unique_features(MUST_HAVE, SELECT_POS_NOLOYAL_COLS)
len(POS_NOLOYAL_FEATURES), POS_NOLOYAL_FEATURES[:5]

In [ ]:
%%time

create_selected_features_table(
    conn=conn_hive,
    output_table='sandbox_crft.mae_nbc_pos_noloyalty_dataset_select_feats_new_scenarios',
    source_table='sandbox_crft.mae_nbc_fm_noloyalty_dataset_new_scenarios',
    features=POS_NOLOYAL_FEATURES,
)

In [ ]:
!rm -rf ./data/datasets/pos/mae_nbc_pos_loyalty_dataset_select_feats_new_scenarios
!rm -rf ./data/datasets/pos/mae_nbc_pos_noloyalty_dataset_select_feats_new_scenarios

In [ ]:
!mkdir -p ./data/datasets/pos

In [ ]:
!hdfs dfs -copyToLocal /warehouse/tablespace/external/hive/sandbox_crft.db/mae_nbc_pos_loyalty_dataset_select_feats_new_scenarios/ ./data/datasets/pos
!hdfs dfs -copyToLocal /warehouse/tablespace/external/hive/sandbox_crft.db/mae_nbc_pos_noloyalty_dataset_select_feats_new_scenarios/ ./data/datasets/pos

In [ ]:
%%time

df_loyalty = load_optimize_save_dataset(
    input_path='./data/datasets/pos/mae_nbc_pos_loyalty_dataset_select_feats_new_scenarios',
    output_csv_path='./data/files/df_loyalty_pos_fs_sel.csv',
)
df_loyalty

In [ ]:
if "df_loyalty" in globals():
    del df_loyalty
gc.collect()

In [ ]:
%%time

df_noloyalty = load_optimize_save_dataset(
    input_path="./data/datasets/pos/mae_nbc_pos_noloyalty_dataset_select_feats_new_scenarios",
    output_csv_path="./data/files/df_noloyalty_pos_fs_sel.csv",
)
df_noloyalty

In [ ]:
del df_noloyalty
gc.collect()

In [ ]:
# Если в память не помещается
# save_parquet_dataset_to_csv_in_batches(
#     input_path="./data/datasets/pos/mae_nbc_pos_noloyalty_dataset_select_feats_new_scenarios",
#     output_csv_path="./data/files/df_noloyalty_pos_fs_sel.csv",
#     batch_size=500_000,
# )

## Bin delta

In [ ]:
# Список выбранных фичей вынесен в configs/feature_lists.json
SELECT_DELTA_LOYAL_COLS = get_feature_list('data_fm_delta_loyalty_columns')
SELECT_DELTA_LOYAL_COLS[:5], len(SELECT_DELTA_LOYAL_COLS)

In [ ]:
DELTA_LOYAL_FEATURES = unique_features(MUST_HAVE, SELECT_DELTA_LOYAL_COLS)
len(DELTA_LOYAL_FEATURES), DELTA_LOYAL_FEATURES[:5]

In [ ]:
%%time

create_selected_features_table(
    conn=conn_hive,
    output_table='sandbox_crft.mae_nbc_delta_bin_loyalty_dataset_select_feats_new_scenarios',
    source_table='sandbox_crft.mae_nbc_fm_loyalty_dataset_new_scenarios',
    features=DELTA_LOYAL_FEATURES,
)

In [ ]:
# Список выбранных фичей вынесен в configs/feature_lists.json
SELECT_DELTA_NOLOYAL_COLS = get_feature_list('data_fm_delta_noloyalty_columns')
SELECT_DELTA_NOLOYAL_COLS[:5], len(SELECT_DELTA_NOLOYAL_COLS)

In [ ]:
DELTA_NOLOYAL_FEATURES = unique_features(MUST_HAVE, SELECT_DELTA_NOLOYAL_COLS)
len(DELTA_NOLOYAL_FEATURES), DELTA_NOLOYAL_FEATURES[:5]

In [ ]:
%%time

create_selected_features_table(
    conn=conn_hive,
    output_table='sandbox_crft.mae_nbc_delta_bin_noloyalty_dataset_select_feats_new_scenarios',
    source_table='sandbox_crft.mae_nbc_fm_noloyalty_dataset_new_scenarios',
    features=DELTA_NOLOYAL_FEATURES,
)

In [ ]:
!rm -rf ./data/datasets/pos_bin_delta/mae_nbc_delta_bin_loyalty_dataset_select_feats_new_scenarios
!rm -rf ./data/datasets/pos_bin_delta/mae_nbc_delta_bin_noloyalty_dataset_select_feats_new_scenarios

In [ ]:
!mkdir -p ./data/datasets/pos_bin_delta

In [ ]:
!hdfs dfs -copyToLocal /warehouse/tablespace/external/hive/sandbox_crft.db/mae_nbc_delta_bin_loyalty_dataset_select_feats_new_scenarios/ ./data/datasets/pos_bin_delta
!hdfs dfs -copyToLocal /warehouse/tablespace/external/hive/sandbox_crft.db/mae_nbc_delta_bin_noloyalty_dataset_select_feats_new_scenarios/ ./data/datasets/pos_bin_delta

In [ ]:
%%time

df_loyalty = load_optimize_save_dataset(
    input_path='./data/datasets/pos_bin_delta/mae_nbc_delta_bin_loyalty_dataset_select_feats_new_scenarios',
    output_csv_path='./data/files/df_loyalty_delta_bin_fs_sel.csv',
)
df_loyalty

In [ ]:
if "df_loyalty" in globals():
    del df_loyalty
gc.collect()

In [ ]:
%%time

df_noloyalty = load_optimize_save_dataset(
    input_path="./data/datasets/pos_bin_delta/mae_nbc_delta_bin_noloyalty_dataset_select_feats_new_scenarios",
    output_csv_path="./data/files/df_noloyalty_delta_bin_fs_sel.csv",
)
df_noloyalty

In [ ]:
del df_noloyalty
gc.collect()

In [ ]:
# %%time
# Если в памяти не помещается
# save_parquet_dataset_to_csv_in_batches(
#     input_path="./data/datasets/pos_bin_delta/mae_nbc_delta_bin_noloyalty_dataset_select_feats_new_scenarios",
#     output_csv_path="./data/files/df_noloyalty_delta_bin_fs_sel.csv",
#     batch_size=500_000,
# )

## Binary

In [ ]:
# Список выбранных фичей вынесен в configs/feature_lists.json
SELECT_BINARY_LOYAL_COLS = get_feature_list('data_fm_binary_loyalty_columns')
SELECT_BINARY_LOYAL_COLS[:5], len(SELECT_BINARY_LOYAL_COLS)

In [ ]:
BINARY_LOYAL_FEATURES = unique_features(MUST_HAVE, SELECT_BINARY_LOYAL_COLS)
len(BINARY_LOYAL_FEATURES), BINARY_LOYAL_FEATURES[:5]

In [ ]:
%%time

create_selected_features_table(
    conn=conn_hive,
    output_table='sandbox_crft.mae_nbc_binary_bin_loyalty_dataset_select_feats_new_scenarios',
    source_table='sandbox_crft.mae_nbc_fm_loyalty_dataset_new_scenarios',
    features=BINARY_LOYAL_FEATURES,
)

In [ ]:
# Список выбранных фичей вынесен в configs/feature_lists.json
SELECT_BINARY_NOLOYAL_COLS = get_feature_list('data_fm_binary_noloyalty_columns')
SELECT_BINARY_NOLOYAL_COLS[:5], len(SELECT_BINARY_NOLOYAL_COLS)

In [ ]:
BINARY_NOLOYAL_FEATURES = unique_features(MUST_HAVE, SELECT_BINARY_NOLOYAL_COLS)
len(BINARY_NOLOYAL_FEATURES), BINARY_NOLOYAL_FEATURES[:5]

In [ ]:
%%time

create_selected_features_table(
    conn=conn_hive,
    output_table='sandbox_crft.mae_nbc_binary_bin_noloyalty_dataset_select_feats_new_scenarios',
    source_table='sandbox_crft.mae_nbc_fm_noloyalty_dataset_new_scenarios',
    features=BINARY_NOLOYAL_FEATURES,
)

In [ ]:
!mkdir ./data/datasets/pos_binary

In [ ]:
!rm -rf ./data/datasets/pos_binary/mae_nbc_binary_bin_loyalty_dataset_select_feats_new_scenarios
!rm -rf ./data/datasets/pos_binary/mae_nbc_binary_bin_noloyalty_dataset_select_feats_new_scenarios

In [ ]:
!hdfs dfs -copyToLocal /warehouse/tablespace/external/hive/sandbox_crft.db/mae_nbc_binary_bin_loyalty_dataset_select_feats_new_scenarios/ ./data/datasets/pos_binary
!hdfs dfs -copyToLocal /warehouse/tablespace/external/hive/sandbox_crft.db/mae_nbc_binary_bin_noloyalty_dataset_select_feats_new_scenarios/ ./data/datasets/pos_binary

In [ ]:
%%time

df_loyalty = load_optimize_save_dataset(
    input_path='./data/datasets/pos_binary/mae_nbc_binary_bin_loyalty_dataset_select_feats_new_scenarios',
    output_csv_path='./data/files/df_loyalty_binary_fs_sel.csv',
)
df_loyalty

In [ ]:
if "df_loyalty" in globals():
    del df_loyalty
gc.collect()

In [ ]:
%%time

df_noloyalty = load_optimize_save_dataset(
    input_path="./data/datasets/pos_binary/mae_nbc_binary_bin_noloyalty_dataset_select_feats_new_scenarios",
    output_csv_path="./data/files/df_noloyalty_binary_fs_sel.csv",
)
df_noloyalty

In [ ]:
# %%time

# save_parquet_dataset_to_csv_in_batches(
#     input_path="./data/datasets/pos_binary/mae_nbc_binary_bin_noloyalty_dataset_select_feats_new_scenarios",
#     output_csv_path="./data/files/df_noloyalty_binary_fs_sel.csv",
#     batch_size=500_000,
# )